# Hafta 9 — Önceden Eğitilmiş Modeller: BERT, GPT, T5

Üç mimari, üç hedef, üç felsefe. Her birini inceliyor, ön eğitim hedeflerini sıfırdan uyguluyor ve modellerin gerçekte ne öğrendiğini görmek için iç temsillerini sorguluyoruz.

## Öğrenme Hedefleri

- Yalnızca kodlayıcı, yalnızca kod çözücü ve kodlayıcı–kod çözücü mimarilerini ve her birinin hangi görevlere uygun olduğunu açıklamak.
- Maskeli dil modellemeyi (MLM) ve nedensel dil modellemeyi uçtan uca uygulamak.
- T5 tarzı span bozmayı (span corruption) uygulamak.
- Bir BERT'i katman katman sorgulamak ve bağlamsal bilginin derinlikle ortaya çıktığı klasik bulguyu yeniden üretmek.

## Zorunlu Okumalar

- Devlin, J. vd. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.
- Radford, A. vd. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2).
- Raffel, C. vd. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5).
- Rogers, A., Kovaleva, O. ve Rumshisky, A. (2020). *A Primer in BERTology*.

In [ ]:
import sys, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1. Bir bakışta üç mimari

| Model | Mimari | Ön eğitim hedefi | En uygun olduğu işler |
|-------|--------------|----------------------|----------|
| **BERT** | Yalnızca kodlayıcı | Maskeli DM + NSP | Sınıflandırma, NER, (çıkarımsal) soru yanıtlama |
| **GPT** | Yalnızca kod çözücü (nedensel) | Sonraki token tahmini | Üretim, bağlam içi öğrenme |
| **T5** | Kodlayıcı–kod çözücü | Span bozma | Her türlü seq2seq görevi (çeviri, özetleme, soru yanıtlama) |

Mimariler yalnızca dikkat maskelerinde farklılaşır:

- BERT: **çift yönlü** — her token diğer her tokena dikkat eder.
- GPT: **nedensel** — $t$ tokenı yalnızca $\leq t$ konumlarına dikkat eder.
- T5: **çift yönlü kodlayıcı + nedensel kod çözücü + çapraz dikkat**.

## 2. Maskeli Dil Modelleme (BERT)

$\mathbf{x}_{1:T}$ girdisi verildiğinde, $|M|/T \approx 0.15$ olacak şekilde rastgele bir $M \subset \{1, \ldots, T\}$ alt kümesi örnekle. Her $i \in M$ için:

- 0,8 olasılıkla $x_i$'yi `[MASK]` ile değiştir;
- 0,1 olasılıkla $x_i$'yi rastgele bir tokenla değiştir;
- 0,1 olasılıkla $x_i$'yi olduğu gibi bırak.

80/10/10 karışımı, eğitim ile çıkarım arasındaki uyumsuzluğu azaltır (çıkarımda `[MASK]` tokenı yoktur). Kayıp *yalnızca maskelenmiş konumlar* üzerinde çapraz entropidir.

$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in M} \log P(x_i \mid \mathbf{x}_{\setminus M}).$$

In [ ]:
# 8. haftada kurduğumuz Transformer kodlayıcısını yeniden kullan (birkaç satır içi basitleştirmeyle).
def scaled_dot(Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    return F.softmax(scores, dim=-1) @ V

class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
    def forward(self, x, mask=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dk).transpose(1, 2)
        out = scaled_dot(q, k, v, mask).transpose(1, 2).contiguous().view(B, T, D)
        return self.o(out)

class Block(nn.Module):
    def __init__(self, d, h, ff, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.attn = MHA(d, h)
        self.ln2 = nn.LayerNorm(d); self.ffn = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.causal = causal
    def forward(self, x, pad_mask=None):
        T = x.size(1)
        mask = None
        if pad_mask is not None:
            mask = pad_mask.unsqueeze(1).unsqueeze(2)
        if self.causal:
            causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
            mask = causal if mask is None else (mask | causal)
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x

class BERT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=256, n_layers=3, max_len=64, pad=0):
        super().__init__()
        self.pad = pad
        self.emb = nn.Embedding(V, d, padding_idx=pad)
        self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, h, ff, causal=False) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)
    def forward(self, x, return_hidden=False):
        T = x.size(1)
        pad_mask = (x == self.pad)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        hiddens = [h]
        for block in self.blocks:
            h = block(h, pad_mask); hiddens.append(h)
        h = self.ln(h)
        logits = self.head(h)
        return (logits, hiddens) if return_hidden else logits

In [ ]:
# Küçük bir dilbilgisine sahip oyuncak bir "dil" (İngilizce sözcüklerle) — MLM'yi iş başında görmeye yeter.
SENTENCES = []
subjects = ['cat', 'dog', 'bird', 'fox', 'student', 'teacher']
verbs    = ['saw', 'chased', 'liked', 'feared', 'ignored', 'followed']
objects  = ['ball', 'book', 'tree', 'house', 'mouse', 'idea']
adv      = ['quickly', 'silently', 'happily', 'reluctantly']

for s in subjects:
    for v in verbs:
        for o in objects:
            SENTENCES.append(f"the {s} {v} the {o}")
            SENTENCES.append(f"the {s} {np.random.choice(adv)} {v} the {o}")

random.shuffle(SENTENCES)
SENTENCES = SENTENCES[:600]

words = sorted(set(w for s in SENTENCES for w in s.split()))
SPECIALS = ['<pad>', '[MASK]', '[CLS]', '[SEP]']
VOCAB = SPECIALS + words
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)
PAD, MASK, CLS, SEP = tok2id['<pad>'], tok2id['[MASK]'], tok2id['[CLS]'], tok2id['[SEP]']
print(f"|V| = {V}, |derlem| = {len(SENTENCES)}")

def encode(sent, max_len=12):
    ids = [CLS] + [tok2id[w] for w in sent.split()] + [SEP]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids

In [ ]:
def make_mlm_batch(sentences, batch_size, mask_rate=0.15, max_len=12):
    sents = random.sample(sentences, batch_size)
    inputs = torch.tensor([encode(s, max_len) for s in sents])
    labels = inputs.clone()
    rand = torch.rand(inputs.shape)
    # Maskeleme için yalnızca özel olmayan tokenları dikkate al.
    eligible = (inputs >= len(SPECIALS))
    mask = (rand < mask_rate) & eligible
    # 80/10/10
    r2 = torch.rand(inputs.shape)
    inputs_masked = inputs.clone()
    inputs_masked[mask & (r2 < 0.8)] = MASK
    rand_pos = mask & (r2 >= 0.8) & (r2 < 0.9)
    inputs_masked[rand_pos] = torch.randint(len(SPECIALS), V, inputs.shape)[rand_pos]
    # Kayıp yalnızca maskelenmiş konumlarda hesaplanır
    labels[~mask] = -100
    return inputs_masked, labels

bert = BERT(V, d=64, h=4, ff=256, n_layers=3, max_len=12, pad=PAD)
opt = torch.optim.Adam(bert.parameters(), lr=3e-3)

losses = []
for step in range(400):
    x, y = make_mlm_batch(SENTENCES, batch_size=32)
    logits = bert(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), ignore_index=-100)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f"adım {step+1:4d}  MLM kaybı = {loss.item():.3f}")

plt.figure(figsize=(8, 3))
plt.plot(losses); plt.xlabel('adım'); plt.ylabel('MLM kaybı')
plt.title('MLM eğitim kaybı'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# İncele: model maskelenmiş bir konumda ne tahmin ediyor?
bert.eval()
sent = "the cat [MASK] the mouse"
ids = [CLS] + [tok2id.get(w, MASK) for w in sent.split()] + [SEP]
ids = ids + [PAD] * (12 - len(ids))
x = torch.tensor([ids])
with torch.no_grad():
    logits = bert(x)
mask_pos = ids.index(MASK)
probs = F.softmax(logits[0, mask_pos], dim=-1)
top = torch.topk(probs, 5)
print(f"Maskeli konum {mask_pos}: '{sent}' için ilk 5 tahmin:")
for p, i in zip(top.values, top.indices):
    print(f"  {id2tok[i.item()]:<12} p = {p.item():.3f}")

## 3. Nedensel Dil Modelleme (GPT)

Aynı Transformer bloğu, öz-dikkatte **nedensel maske**. Kayıp, her konum üzerinde sonraki token tahminidir:

$$\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{T-1} \log P(x_{t+1} \mid x_{\leq t}).$$

`[MASK]` tokenı yok, yardımcı hedef yok, özel bir hesap kitap yok. GPT'nin bu kadar iyi ölçeklenmesinin bir nedeni bu sadeliktir.

In [ ]:
class GPT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=256, n_layers=3, max_len=16, pad=0):
        super().__init__()
        self.pad = pad
        self.emb = nn.Embedding(V, d, padding_idx=pad)
        self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, h, ff, causal=True) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)
    def forward(self, x):
        T = x.size(1)
        pad_mask = (x == self.pad)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h, pad_mask)
        return self.head(self.ln(h))

def make_clm_batch(sentences, batch_size, max_len=14):
    sents = random.sample(sentences, batch_size)
    seqs = []
    for s in sents:
        ids = [CLS] + [tok2id[w] for w in s.split()] + [SEP]
        ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
        seqs.append(ids)
    seqs = torch.tensor(seqs)
    return seqs[:, :-1], seqs[:, 1:]

gpt = GPT(V, d=64, h=4, ff=256, n_layers=3, max_len=14, pad=PAD)
opt = torch.optim.Adam(gpt.parameters(), lr=3e-3)
for step in range(400):
    x, y = make_clm_batch(SENTENCES, batch_size=32)
    logits = gpt(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 100 == 0:
        print(f"adım {step+1:4d}  CLM kaybı = {loss.item():.3f}")

In [ ]:
# Üretim
gpt.eval()
prompt = "[CLS] the cat"
ids = [tok2id[w] for w in prompt.split() if w in tok2id]
for _ in range(8):
    x = torch.tensor([ids + [PAD] * (13 - len(ids))])
    with torch.no_grad():
        logits = gpt(x)
    next_id = torch.multinomial(F.softmax(logits[0, len(ids) - 1] / 0.7, dim=-1), 1).item()
    if next_id == SEP or next_id == PAD:
        break
    ids.append(next_id)
print("Üretilen:", ' '.join(id2tok[i] for i in ids if i not in (PAD,)))

## 4. T5 tarzı span bozma

T5 (Raffel vd., 2020) tek bir metinden-metne hedef kullanır. Ön eğitim sırasında girdideki ardışık token span'leri `<extra_id_0>`, `<extra_id_1>`, ... gibi gözcü (sentinel) tokenlarla değiştirilir ve model, aynı gözcülerle ayrılmış *eksik span'leri üretmek* üzere eğitilir.

> **Girdi:**  the cat `<X>` the `<Y>` mouse
> **Hedef:** `<X>` chased `<Y>` little `<Z>`

Bu kavramsal olarak MLM'dir ama seq2seq üretimi olarak çerçevelenmiştir; bu da T5'in her görevi tek tip ele almasını sağlar. Aşağıdaki kodda gözcüler kısaca `<X0>`, `<X1>`, ... olarak yazılır. Kısa cümlelerde hiçbir span seçilmeyebileceğinden, her örnekte en az bir span'in bozulması garanti edilir.

In [ ]:
# Yalnızca veri hazırlığı — bir kodlayıcı-kod çözücü ön eğitim döngüsü çalıştırmak,
# bu maskeleme uygulanmış hâliyle yapısal olarak 8. haftanın aynısıdır.
SENTINEL_BASE = V  # kavramsal sözlüğü genişletiyoruz
def make_span_corruption_example(sent, span_prob=0.15, mean_span=2):
    tokens = sent.split()
    n = len(tokens)
    keep = np.ones(n, dtype=bool)
    i = 0
    while i < n:
        if np.random.random() < span_prob:
            span_len = max(1, int(np.random.exponential(mean_span)))
            keep[i:i+span_len] = False
            i += span_len
        else:
            i += 1
    if keep.all():
        # Kısa cümlelerde hiçbir span seçilmemiş olabilir: en az bir span'i boz.
        start = np.random.randint(0, n)
        span_len = max(1, int(np.random.exponential(mean_span)))
        keep[start:start + span_len] = False
    # Kaynağı (gözcülerle) ve hedefi (gözcü + atılan span'ler) kur.
    src_tokens, tgt_tokens = [], []
    sentinel_idx = 0
    i = 0
    while i < n:
        if keep[i]:
            src_tokens.append(tokens[i]); i += 1
        else:
            src_tokens.append(f"<X{sentinel_idx}>")
            tgt_tokens.append(f"<X{sentinel_idx}>")
            while i < n and not keep[i]:
                tgt_tokens.append(tokens[i]); i += 1
            sentinel_idx += 1
    tgt_tokens.append(f"<X{sentinel_idx}>")
    return ' '.join(src_tokens), ' '.join(tgt_tokens)

for sent in random.sample(SENTENCES, 3):
    src, tgt = make_span_corruption_example(sent)
    print(f"orijinal: {sent}")
    print(f"girdi   : {src}")
    print(f"hedef   : {tgt}\n")

## 5. Sondalama — BERT ne öğreniyor?

Doğrusal bir sonda (probe): BERT'i dondur, her katmanın temsilleri üzerinde küçük bir doğrusal sınıflandırıcı eğiterek bir dilbilimsel özelliği (ör. sözcük türü) tahmin et. Doğruluk $\ell$ katmanında zirve yapıyorsa, model o özelliği en temiz biçimde orada temsil ediyordur.

Klasik bulgu (Tenney vd., 2019; Rogers vd., 2020): **BERT dilbilimsel boru hattını sırayla yeniden kurar** — altta yüzey özellikleri, ortada sözdizimi, üste yakın anlambilim.

İki sonda çalıştırıyoruz:

1. **Kontrol sondası — sözcük sınıfı.** Bu oyuncak dilbilgisinde her sözcüğün sınıfı yalnızca kimliğinden belirlenir (*cat* her zaman özne, *chased* her zaman fiildir). Bu yüzden gömme katmanı dâhil **her katmanda ~%100** bekleriz; böyle bir sonda katmanlar arasında hiçbir şey ayırt edemez. Bu, yüksek sonda doğruluğunun tek başına "model bu özelliği öğrendi" anlamına gelmediğini hatırlatır (Hewitt ve Liang, 2019 — *control tasks*).
2. **Bağlamsal sonda — öznenin türü.** Yalnızca **fiil** konumundaki temsilden, cümlenin öznesinin bir hayvan mı (*cat, dog, bird, fox*) yoksa bir insan mı (*student, teacher*) olduğunu tahmin ederiz. Fiil sözcüğü ve konumu bu bilgiyi taşımaz; bilgi ancak dikkat yoluyla özneden taşınabilir. Bu yüzden 0. katmanın çoğunluk sınıfı düzeyinde (~0,67), derin katmanların bunun belirgin biçimde üzerinde olmasını bekleriz. Sonda ayrılmış bir test bölümünde değerlendirilir.

In [ ]:
# Her sözcüğü bir kategoriyle etiketle: ÖZNE, FİİL, NESNE, ZARF, BELİRLEYİCİ, DİĞER.
TAG = {}
for w in subjects: TAG[w] = 0   # ÖZNE
for w in verbs:    TAG[w] = 1   # FİİL
for w in objects:  TAG[w] = 2   # NESNE
for w in adv:      TAG[w] = 3   # ZARF
TAG['the'] = 4                  # BELİRLEYİCİ
for w in SPECIALS: TAG[w] = 5   # DİĞER

bert.eval()
# Gizli durumları katman katman ve doğru etiketleriyle birlikte topla.
hiddens_by_layer = {l: [] for l in range(len(bert.blocks) + 1)}
tags = []
with torch.no_grad():
    for sent in SENTENCES[:200]:
        ids = encode(sent)
        x = torch.tensor([ids])
        _, hs = bert(x, return_hidden=True)
        for l, h in enumerate(hs):
            for t in range(len(ids)):
                if ids[t] != PAD:
                    hiddens_by_layer[l].append(h[0, t].numpy())
        for t in range(len(ids)):
            if ids[t] != PAD:
                tags.append(TAG[id2tok[ids[t]]])

tags = np.array(tags)

# Basit doğrusal sonda: kapalı biçimli en küçük kareler sınıflandırıcısı.
from numpy.linalg import lstsq
def probe_accuracy(H, y, n_classes=6):
    Y = np.eye(n_classes)[y]
    H_bias = np.hstack([H, np.ones((len(H), 1))])
    W, *_ = lstsq(H_bias, Y, rcond=None)
    preds = (H_bias @ W).argmax(axis=1)
    return (preds == y).mean()

print("1) Kontrol sondası — sözcük sınıfı (sözcük kimliğinden belirlenir):")
accs = []
for l, hs in hiddens_by_layer.items():
    H = np.stack(hs)
    accs.append(probe_accuracy(H, tags))
    print(f"   katman {l}: sonda doğruluğu = {accs[-1]:.3f}")

# 2) Bağlamsal sonda: fiil konumundaki temsilden öznenin türünü (hayvan / insan) tahmin et.
ANIMALS = {'cat', 'dog', 'bird', 'fox'}
verb_h = {l: [] for l in range(len(bert.blocks) + 1)}
subj_is_animal = []
with torch.no_grad():
    for sent in SENTENCES:
        w = sent.split()
        ids = encode(sent)
        v_pos = 1 + next(i for i, t in enumerate(w) if t in verbs)   # +1: [CLS]
        _, hs = bert(torch.tensor([ids]), return_hidden=True)
        for l, h in enumerate(hs):
            verb_h[l].append(h[0, v_pos].numpy())
        subj_is_animal.append(int(w[1] in ANIMALS))
subj_is_animal = np.array(subj_is_animal)

perm = np.random.permutation(len(subj_is_animal))
tr, te = perm[: int(0.7 * len(perm))], perm[int(0.7 * len(perm)):]
def heldout_probe(H, y, n_classes=2):
    Hb = np.hstack([H, np.ones((len(H), 1))])
    W, *_ = lstsq(Hb[tr], np.eye(n_classes)[y[tr]], rcond=None)
    return ((Hb[te] @ W).argmax(axis=1) == y[te]).mean()

majority = max(subj_is_animal.mean(), 1 - subj_is_animal.mean())
print(f"\n2) Bağlamsal sonda — fiil konumundan öznenin türü (çoğunluk sınıfı düzeyi = {majority:.3f}):")
ctx_accs = []
for l in verb_h:
    ctx_accs.append(heldout_probe(np.stack(verb_h[l]), subj_is_animal))
    print(f"   katman {l}: test doğruluğu = {ctx_accs[-1]:.3f}")

layers = list(hiddens_by_layer.keys())
plt.figure(figsize=(7, 4))
plt.plot(layers, accs, marker='o', label='sözcük sınıfı (kontrol)')
plt.plot(layers, ctx_accs, marker='s', label='öznenin türü, fiil konumundan (bağlamsal)')
plt.axhline(majority, color='gray', ls='--', lw=1, label='çoğunluk sınıfı düzeyi')
plt.xlabel('katman (0 = gömme)'); plt.ylabel('sonda doğruluğu')
plt.title('Doğrusal sonda doğruluğu ve katman derinliği')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Üç hedefi sabit bir bütçe altında karşılaştırmak

Yararlı bir deney (Liu vd., 2019; Raffel vd., 2020): parametreleri, eğitim tokenlarını ve mimariyi sabit tut, yalnızca ön eğitim hedefini değiştir. Literatürdeki uzlaşı:

- **MLM**, anlama görevleri (GLUE, SuperGLUE) için en iyisidir.
- **CLM**, üretim için en iyisidir ve model boyutuyla en güçlü ölçeklenmeyi gösterir.
- **Span bozma**, MLM'ye denk gelir veya onu geçer; ayrıca MLM'den farklı olarak önceden eğitilmiş model, mimari değişikliği gerektirmeden *doğrudan* her görevde kullanılabilir.

Modern öncü modeller, mühendislik nedenleriyle ağırlıklı olarak CLM'dir (yalnızca kod çözücü): daha basittir, uzun bağlamlara ölçeklenir ve çıkarımda KV önbelleğine izin verir.

## 7. Alıştırmalar

1. **Küçük bir BERT'i önceden eğitin ve GLUE tarzı aktarımı ölçün.** 1M cümle üzerinde 6 katmanlı bir model önceden eğitin, ardından 1k, 10k ve 100k örnekle ikili bir duygu görevinde ince ayar yapın. İnce ayarlı doğruluğu ön eğitim adımlarına göre çizdirin.
2. **Cümle benzerliğinde GPT ve BERT.** GPT cümle gömmeleri için tasarlanmamıştır, ama ortalama havuzlamayla yine de bunları üretebilir. İkisini karşılaştırın; ardından BERT'in neden genellikle sıfır atışta kazandığını, ama GPT'nin istemlerle neden yetişebildiğini açıklayın (Reimers ve Gurevych, 2019; Jiang vd., 2022).
3. **Span bozma ablasyonu.** T5 tarzı bozmada span uzunluğunu (geometrik ortalama $\in \{1, 3, 5, 10\}$) değiştirin. Alt görev aktarımını ölçün. Ortalama span = 3 (T5'in varsayılanı) deneysel olarak nereden geliyor?
4. **Sondalama katman sırası.** Tenney vd. (2019)'u yeniden üretin: sözcük türü, bağımlılık, NER ve eşgönderim için sondalayın. Her görev için en iyi katmanın kabaca bu sırayla arttığını doğrulayın. Her sonda için bir kontrol görevi tanımlayıp *seçicilik* ölçün (Hewitt ve Liang, 2019).

---

## Gelecek Hafta

Hafta 10 — İnce ayar ve hizalama. Tam ince ayar, LoRA, RLHF ve DPO.